# Relationship discovery from data alone

Scorecard dimension D1b. The analyst supplies only the data and the response
name. Each method reports the predictors it finds, and the score counts the
planted predictors it recovers (true positive rate, TPR) and the reported
predictors that are not planted (false discovery rate, FDR).

Protocol: `discovery_d1b/PROTOCOL.md`. Results: `discovery_d1b/results.json`,
scores from `discovery_d1b/score.py`.

## Methods

- `mboostrs_discover`: the formula from `mb.discover`, default settings.
- `catboost_importance` and `lightgbm_importance`: predictors whose importance
  exceeds the 95th percentile of the same model fitted to permuted responses.

Three processes, 10 seeds each, 8 candidate predictors, 500 rows per draw.

In [1]:
import json
from pathlib import Path

import pandas as pd

root = Path("discovery_d1b")
scores = json.loads((root / "scores.json").read_text())
rows = []
for process, methods in scores.items():
    for method, e in methods.items():
        rows.append({
            "process": process,
            "method": method,
            "TPR": round(e["tpr"], 2) if "tpr" in e else None,
            "FDR": round(e["fdr"], 2) if e["fdr"] is not None else None,
            "draws with any selection (noise)": round(e["any_selection_rate"], 2) if "any_selection_rate" in e else None,
        })
pd.DataFrame(rows)

,process,method,TPR,FDR,draws with any selection (noise)
0,additive_nonlinear,catboost_importance,0.55,0.00,NaN
1,additive_nonlinear,lightgbm_importance,0.85,0.00,NaN
2,additive_nonlinear,mboostrs_discover,1.00,0.26,NaN
3,interaction,catboost_importance,1.00,0.00,NaN
4,interaction,lightgbm_importance,1.00,0.00,NaN
5,interaction,mboostrs_discover,1.00,0.00,NaN
6,pure_noise,catboost_importance,NaN,1.00,0.2
7,pure_noise,lightgbm_importance,NaN,1.00,0.5
8,pure_noise,mboostrs_discover,NaN,NaN,0.0


## Reading the table

- **Additive process.** mboostrs recovers both planted predictors in every
  draw (TPR 1.00), against 0.55 for CatBoost and 0.85 for LightGBM. But its FDR
  is 0.26, while both competitors have FDR 0. The false predictors enter through
  the interaction search (issue #565). This is the gap to the best method for
  this process: the FDR, not the recall.
- **Interaction process.** All three methods recover both planted predictors
  (TPR 1.00). The interaction search is required here: with it switched off,
  mboostrs recovers none (see #565).
- **Pure noise.** mboostrs reports nothing in any draw. CatBoost and LightGBM
  each report something in some draws (20% and 50%), so their FDR is 1 on
  this process.